## Bayesian methods of hyperparameter optimization

In addition to the random search and the grid search methods for selecting optimal hyperparameters, we can use Bayesian methods of probabilities to select the optimal hyperparameters for an algorithm.

In this case study, we will be using the BayesianOptimization library to perform hyperparameter tuning. This library has very good documentation which you can find here: https://github.com/fmfn/BayesianOptimization

You will need to install the Bayesian optimization module. Running a cell with an exclamation point in the beginning of the command will run it as a shell command — please do this to install this module from our notebook in the cell below.

In [1]:
#! pip install bayesian-optimization lightgbm catboost

In [2]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import lightgbm
from bayes_opt import BayesianOptimization
from catboost import CatBoostClassifier, cv, Pool

In [3]:
print(lightgbm.__version__)

4.6.0


In [4]:
import os
os.listdir()

['Bayesian_optimization_case_study_ams.ipynb',
 '.DS_Store',
 'Optuna_Bayesian_optimization_case_study.ipynb',
 '.github',
 'data']

## How does Bayesian optimization work?

Bayesian optimization works by constructing a posterior distribution of functions (Gaussian process) that best describes the function you want to optimize. As the number of observations grows, the posterior distribution improves, and the algorithm becomes more certain of which regions in parameter space are worth exploring and which are not, as seen in the picture below.

<img src="https://github.com/fmfn/BayesianOptimization/blob/master/examples/bo_example.png?raw=true" />
As you iterate over and over, the algorithm balances its needs of exploration and exploitation while taking into account what it knows about the target function. At each step, a Gaussian Process is fitted to the known samples (points previously explored), and the posterior distribution, combined with an exploration strategy (such as UCB — aka Upper Confidence Bound), or EI (Expected Improvement). This process is used to determine the next point that should be explored (see the gif below).
<img src="https://github.com/fmfn/BayesianOptimization/raw/master/examples/bayesian_optimization.gif" />

## Let's look at a simple example

The first step is to create an optimizer. It uses two items:
* function to optimize
* bounds of parameters

The function is the procedure that counts metrics of our model quality. The important thing is that our optimization will maximize the value on function. Smaller metrics are best. Hint: don't forget to use negative metric values.

Here we define our simple function we want to optimize.

In [5]:
def simple_func(a, b):
    return a + b

Now, we define our bounds of the parameters to optimize, within the Bayesian optimizer.

The main parameters of the BayesianOptimization function are:

1. The function to optimize (e.g., simple_func)
2. The parameter bounds (e.g., {'a': (1, 3), 'b': (4, 7)})

In [6]:
# Initialize BayesianOptimization object

optimizer = BayesianOptimization(
    simple_func, # Add function to optimize
    {'a': (1, 3), # Define search space for parameter bounds of 'a' and 'b'
    'b': (4, 7)})

# Optimizer will use Bayesian optimization to search for the combination of 'a' and 'b' that gives the highest value of ('maximizes') simple_func(a, b)

The main arguments used when you call the maximize() method on the optimizer object:

* **n_iter:** This is how many steps of Bayesian optimization you want to perform. The more steps, the more likely you are to find a good maximum.

n_iter sets how many times the optimizer will try new parameter values after the initial random explorations. Each step, it uses what it has learned so far to pick the next best combination to test. 

* **init_points:** This is how many steps of random exploration you want to perform. Random exploration can help by diversifying the exploration space.

This is the number of initial random explorations before Bayesian optimization starts. These initial random trials help the optimizer gather information about the search space, so it can build a better model of how the parameters affect the function. 

**Let's run an example where we use the optimizer to find the best values to maximize the target value for a and b given the inputs of 3 and 2.**

In [7]:
optimizer.maximize(3,2)

|   iter    |  target   |     a     |     b     |
-------------------------------------------------
| 1         | 8.5112973 | 1.9047834 | 6.6065139 |
| 2         | 5.9994646 | 1.0090904 | 4.9903741 |
| 3         | 6.0769414 | 1.8205997 | 4.2563417 |
| 4         | 9.6792025 | 2.6792025 | 7.0       |
| 5         | 9.1996454 | 3.0       | 6.1996454 |


Great, now let's print the best parameters and the associated maximized target.

In [8]:
print(optimizer.max['params']);optimizer.max['target']

{'a': np.float64(2.6792025508532276), 'b': np.float64(7.0)}


np.float64(9.679202550853228)

## Test it on real data using the Light GBM

The dataset we will be working with is the famous flight departures dataset. Our modeling goal will be to predict if a flight departure is going to be delayed by 15 minutes based on the other attributes in our dataset. As part of this modeling exercise, we will use Bayesian hyperparameter optimization to identify the best parameters for our model.

**<font color='teal'> You can load the zipped csv files just as you would regular csv files using Pandas read_csv. In the next cell load the train and test data into two seperate dataframes. </font>**


In [9]:
train_df = pd.read_csv('data/flight_delays_train.csv')
test_df = pd.read_csv('data/flight_delays_test.csv')

**<font color='teal'> Print the top five rows of the train dataframe and review the columns in the data. </font>**

In [10]:
train_df.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423,Y


**<font color='teal'> Use the describe function to review the numeric columns in the train dataframe. </font>**

In [11]:
train_df.describe()

,DepTime,Distance
count,100000.000000,100000.00000
mean,1341.523880,729.39716
std,476.378445,574.61686
min,1.000000,30.00000
25%,931.000000,317.00000
50%,1330.000000,575.00000
75%,1733.000000,957.00000
max,2534.000000,4962.00000


Notice, `DepTime` is the departure time in a numeric representation in 2400 hours. 

 **<font color='teal'>The response variable is 'dep_delayed_15min' which is a categorical column, so we need to map the Y for yes and N for no values to 1 and 0. Run the code in the next cell to do this.</font>**

In [12]:
train_df = train_df[train_df.DepTime <= 2400].copy()
y_train = train_df['dep_delayed_15min'].map({'Y': 1, 'N': 0}).values

## Feature Engineering
Use these defined functions to create additional features for the model. Run the cell to add the functions to your workspace.

In [13]:
# Define a function to encode categorical values as integers.
# LabelEncoder().fit_transform(df_column) fits a label encoder to the column and transforms each unique value to an integer.
# Returns the encoded column.

def label_enc(df_column):
    df_column = LabelEncoder().fit_transform(df_column)
    return df_column

In [14]:
# Multiply the input value by 2π/period to scale it to a full cycle.
# Returns the sine of the scaled value (useful for cyclical features like time).

def make_harmonic_features_sin(value, period=2400):
    value *= 2 * np.pi / period 
    return np.sin(value)

# Same as above, but returns the cosine of the scaled value.

def make_harmonic_features_cos(value, period=2400):
    value *= 2 * np.pi / period 
    return np.cos(value)

In [15]:

def feature_eng(df):
    # Concatenate the origin and destination airport codes.
    df['flight'] = df['Origin']+df['Dest']

    # Extract month number from string (in this case, c-##) and convert to integer.
    df['Month'] = df.Month.map(lambda x: x.split('-')[-1]).astype('int32')

    # Extract day number from string and convert to integer.
    df['DayofMonth'] = df.DayofMonth.map(lambda x: x.split('-')[-1]).astype('uint8')

    # Create binary features for whether the day is at the beginning, middle, or end of the month.
    df['begin_of_month'] = (df['DayofMonth'] < 10).astype('uint8')
    df['midddle_of_month'] = ((df['DayofMonth'] >= 10)&(df['DayofMonth'] < 20)).astype('uint8')
    df['end_of_month'] = (df['DayofMonth'] >= 20).astype('uint8')

    # Extract day of week number from string and convert to integer.
    df['DayOfWeek'] = df.DayOfWeek.map(lambda x: x.split('-')[-1]).astype('uint8')

    # Convert departure time (e.g., 1300) to hour (e.g., 13).
    df['hour'] = df.DepTime.map(lambda x: x/100).astype('int32')

    # Create binary features for different times of day.
    df['morning'] = df['hour'].map(lambda x: 1 if (x <= 11)& (x >= 7) else 0).astype('uint8')
    df['day'] = df['hour'].map(lambda x: 1 if (x >= 12) & (x <= 18) else 0).astype('uint8')
    df['evening'] = df['hour'].map(lambda x: 1 if (x >= 19) & (x <= 23) else 0).astype('uint8')
    df['night'] = df['hour'].map(lambda x: 1 if (x >= 0) & (x <= 6) else 0).astype('int32')

    # Create binary features for each season based on month.
    df['winter'] = df['Month'].map(lambda x: x in [12, 1, 2]).astype('int32')
    df['spring'] = df['Month'].map(lambda x: x in [3, 4, 5]).astype('int32')
    df['summer'] = df['Month'].map(lambda x: x in [6, 7, 8]).astype('int32')
    df['autumn'] = df['Month'].map(lambda x: x in [9, 10, 11]).astype('int32')

    # Create binary features for holidays (Friday (5), Saturday (6), Sunday (7)) and weekdays.
    df['holiday'] = (df['DayOfWeek'] >= 5).astype(int) 
    df['weekday'] = (df['DayOfWeek'] < 5).astype(int)

    # Add count-based features: number of flights per destination/origin per month, total flights per destination/origin, and flights per carrier (overall and per month).
    df['airport_dest_per_month'] = df.groupby(['Dest', 'Month'])['Dest'].transform('count')
    df['airport_origin_per_month'] = df.groupby(['Origin', 'Month'])['Origin'].transform('count')
    df['airport_dest_count'] = df.groupby(['Dest'])['Dest'].transform('count')
    df['airport_origin_count'] = df.groupby(['Origin'])['Origin'].transform('count')
    df['carrier_count'] = df.groupby(['UniqueCarrier'])['Dest'].transform('count')
    df['carrier_count_per month'] = df.groupby(['UniqueCarrier', 'Month'])['Dest'].transform('count')

    # Create cyclical (harmonic) features for departure time.
    df['deptime_cos'] = df['DepTime'].map(make_harmonic_features_cos)
    df['deptime_sin'] = df['DepTime'].map(make_harmonic_features_sin)

    # Create combined features by concatenation of flight, destination, origin, and carrier.
    df['flightUC'] = df['flight']+df['UniqueCarrier']
    df['DestUC'] = df['Dest']+df['UniqueCarrier']
    df['OriginUC'] = df['Origin']+df['UniqueCarrier']

    # Drop the original 'DepTime' column because its information is now captured in new features. Keeping it would be redundant and could lead to multicollinearity.
    return df.drop('DepTime', axis=1)

Concatenate by stacking (the default) the training and testing dataframes and then apply the earlier defined feature engineering functions to the full dataframe by calling the feature_eng() function.

In [16]:
full_df = pd.concat([train_df.drop('dep_delayed_15min', axis=1), test_df])
full_df = feature_eng(full_df)

In [17]:
full_df.head()

,Month,DayofMonth,DayOfWeek,UniqueCarrier,Origin,Dest,Distance,flight,begin_of_month,midddle_of_month,...,airport_origin_per_month,airport_dest_count,airport_origin_count,carrier_count,carrier_count_per month,deptime_cos,deptime_sin,flightUC,DestUC,OriginUC
0,8,21,7,AA,ATL,DFW,732,ATLDFW,0,0,...,1016,8290,11375,18024,1569,0.343660,-0.939094,ATLDFWAA,DFWAA,ATLAA
1,4,20,3,US,PIT,MCO,834,PITMCO,0,0,...,105,3523,1390,13069,1094,-0.612907,-0.790155,PITMCOUS,MCOUS,PITUS
2,9,2,5,XE,RDU,CLE,416,RDUCLE,1,0,...,136,2246,1747,11737,977,-0.835807,-0.549023,RDUCLEXE,CLEXE,RDUXE
3,11,25,6,OO,DEN,MEM,872,DENMEM,0,0,...,514,1785,6222,15343,1242,-0.884988,0.465615,DENMEMOO,MEMOO,DENOO
4,10,7,6,WN,MDW,OMA,423,MDWOMA,1,0,...,226,687,2571,30958,2674,0.073238,-0.997314,MDWOMAWN,OMAWN,MDWWN


In [18]:
# This code block applies the label_enc function to categorical columns in the full_df DataFrame. 'label_enc' converts categorical string values into integer labels, which are easier for machine learning models to process.

for column in ['UniqueCarrier', 'Origin', 'Dest','flight',  'flightUC', 'DestUC', 'OriginUC']:
    full_df[column] = label_enc(full_df[column])

In [19]:
full_df

,Month,DayofMonth,DayOfWeek,UniqueCarrier,Origin,Dest,Distance,flight,begin_of_month,midddle_of_month,...,airport_origin_per_month,airport_dest_count,airport_origin_count,carrier_count,carrier_count_per month,deptime_cos,deptime_sin,flightUC,DestUC,OriginUC
0,8,21,7,1,19,82,732,171,0,0,...,1016,8290,11375,18024,1569,0.343660,-0.939094,265,494,67
1,4,20,3,19,226,180,834,3986,0,0,...,105,3523,1390,13069,1094,-0.612907,-0.790155,6907,1085,1441
2,9,2,5,21,239,62,416,4091,1,0,...,136,2246,1747,11737,977,-0.835807,-0.549023,7064,359,1518
3,11,25,6,16,81,184,872,1304,0,0,...,514,1785,6222,15343,1242,-0.884988,0.465615,2258,1122,484
4,10,7,6,20,182,210,423,2979,1,0,...,226,687,2571,30958,2674,0.073238,-0.997314,5144,1313,1103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,6,5,2,20,71,132,187,1002,1,0,...,16,1513,224,30958,2558,-0.612907,0.790155,1680,766,413
99996,11,24,6,18,213,159,1515,3635,0,0,...,773,4920,9823,13311,1065,-0.799685,-0.600420,6205,955,1331
99997,1,30,2,16,213,260,438,3689,0,0,...,795,278,9823,15343,1149,-0.690251,-0.723570,6331,1719,1330
99998,1,5,5,7,166,19,761,2685,1,0,...,283,11379,3350,14624,1291,-0.509041,0.860742,4650,71,994


Split the full_df back into training and testing sets based on the original training set size. 

In [20]:
# Select the first N rows of full_df, where N is the number of rows in the original training set (train_df.shape[0]). These rows correspond to the training data.
X_train = full_df[:train_df.shape[0]]

# Select all rows from position N onward, which correspond to the test data.
X_test = full_df[train_df.shape[0]:]

Create a list of the categorical features.

In [21]:
categorical_features = ['Month',  'DayOfWeek', 'UniqueCarrier', 'Origin', 'Dest','flight',  'flightUC', 'DestUC', 'OriginUC']

Let's build a light GBM model to test the bayesian optimizer.

### [LightGBM](https://lightgbm.readthedocs.io/en/latest/) is a gradient boosting framework that uses tree-based learning algorithms. It is designed to be distributed and efficient with the following advantages:

* Faster training speed and higher efficiency.
* Lower memory usage.
* Better accuracy.
* Support of parallel and GPU learning.
* Capable of handling large-scale data.

First, we define the function we want to maximize and that will count cross-validation metrics of lightGBM for our parameters.

Some params such as num_leaves, max_depth, min_child_samples, min_data_in_leaf should be integers.

In [22]:
def lgb_eval(num_leaves,max_depth,lambda_l2,lambda_l1,min_child_samples, min_data_in_leaf):
    params = {
        "objective" : "binary",
        "metric" : "auc", 
        'is_unbalance': True,
        "num_leaves" : int(num_leaves),
        "max_depth" : int(max_depth),
        "lambda_l2" : lambda_l2,
        "lambda_l1" : lambda_l1,
        "num_threads" : 20,
        "min_child_samples" : int(min_child_samples),
        'min_data_in_leaf': int(min_data_in_leaf),
        "learning_rate" : 0.03,
        "subsample_freq" : 5,
        "bagging_seed" : 42,
        "verbosity" : -1
    }
    lgtrain = lightgbm.Dataset(X_train, y_train,categorical_feature=categorical_features)
    cv_result = lightgbm.cv(params,
                       lgtrain,
                       num_boost_round=1000,
                       stratified=True,
                       nfold=3,
                          callbacks=[lightgbm.early_stopping(stopping_rounds=50)]
                       )

    return cv_result['valid auc-mean'][-1]

Apply the Bayesian optimizer to the function we created in the previous step to identify the best hyperparameters. We will run 5 iterations and set init_points = 2.


In [23]:
# This instantiates the BayesianOptimization object and sets up the optimization problem. It defines the hyperparameter search space for LightGBM and specifies the evaluation function (lgb_eval) to maximize. It does not run the optimization yet.
lgbBO = BayesianOptimization(
    lgb_eval,
    {
        'num_leaves': (25, 512),           # reduced from 4000
        'max_depth': (5, 32),              # reduced from 63
        'lambda_l2': (0.0, 0.05),
        'lambda_l1': (0.0, 0.05),
        'min_child_samples': (50, 500),    # reduced from 10000
        'min_data_in_leaf': (100, 500)     # reduced from 2000
    }
)

# Run the Bayesian optimization process to find the best hyperparameters for LightGBM. It will perform 2 initial random evaluations followed by 5 iterations of optimization based on the results of previous evaluations.
lgbBO.maximize(n_iter=5, init_points=2)

|   iter    |  target   | num_le... | max_depth | lambda_l2 | lambda_l1 | min_ch... | min_da... |
-------------------------------------------------------------------------------------------------
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[26]	valid's auc: 0.721452 + 0.00277963
| 1         | 0.7214521 | 460.60366 | 22.572886 | 0.0299828 | 0.0039377 | 247.35686 | 190.29713 |
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[24]	valid's auc: 0.718704 + 0.00258608
| 2         | 0.7187037 | 102.07482 | 31.269049 | 0.0225485 | 0.0130584 | 99.894007 | 324.93183 |
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[30]	valid's auc: 0.721069 + 0.00224617
| 3         | 0.7210688 | 471.98590 | 16.232278 | 0.0124666 | 0.0402145 | 245.92713 | 180.80155 |
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[27]	v

 **<font color='teal'> Print the best result by using the '.max' function.</font>**

In [24]:
lgbBO.max

{'target': np.float64(0.721703584117023),
 'params': {'num_leaves': np.float64(428.23383905332673),
  'max_depth': np.float64(28.422081999516788),
  'lambda_l2': np.float64(0.043140807137549336),
  'lambda_l1': np.float64(0.02887893281767975),
  'min_child_samples': np.float64(282.4698438817827),
  'min_data_in_leaf': np.float64(161.00373984041215)}}

Review the process at each step by using the '.res[0]' function.

In [25]:
lgbBO.res[0]

{'target': np.float64(0.7214521684702359),
 'params': {'num_leaves': np.float64(460.60366049783795),
  'max_depth': np.float64(22.572886754004095),
  'lambda_l2': np.float64(0.029982866368215844),
  'lambda_l1': np.float64(0.003937768838968159),
  'min_child_samples': np.float64(247.35686208453905),
  'min_data_in_leaf': np.float64(190.29713129987005)}}

In [ ]:
# Split your training data for validation
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Extract best parameters and ensure integer parameters are cast correctly
best_params = lgbBO.max['params']
best_params['num_leaves'] = int(best_params['num_leaves'])
best_params['max_depth'] = int(best_params['max_depth'])
best_params['min_child_samples'] = int(best_params['min_child_samples'])
best_params['min_data_in_leaf'] = int(best_params['min_data_in_leaf'])

# Create and fit the model
final_model = lightgbm.LGBMClassifier(
    **best_params,
    n_estimators=1000,
    learning_rate=0.03,
    objective="binary",
    is_unbalance=True
)

final_model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val)],
    categorical_feature=categorical_features,
    callbacks=[lightgbm.early_stopping(50)]
)


Python(17024) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[11]	valid_0's binary_logloss: 0.466684


,boosting_type,'gbdt'
,num_leaves,428
,max_depth,28
,learning_rate,0.03
,n_estimators,1000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,282


In [28]:
# Get feature importances

feature_importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_model.feature_importances_
})

# Sort by importance and display the top features
feature_importance_df.sort_values(by='importance', ascending=False).head(10)

,feature,importance
28,deptime_cos,409
6,Distance,367
29,deptime_sin,308
23,airport_origin_per_month,300
27,carrier_count_per month,299
22,airport_dest_per_month,292
1,DayofMonth,283
4,Origin,178
5,Dest,177
25,airport_origin_count,171
